In [1]:
import os
import sys
import pandas as pd
sys.path.append('..')

from src.vectorstore import csv_loader, build_vectorstore
from src.rag_pipeline import load_llm, semantic_retriever, initialize_rag_chain, format_docs_to_df
from src.hybrid import preprocess_query, bm25_retriever, hybrid_retriever
from src.prompts import prompt, prompt_no_instruct, prompt_concise
from IPython.display import Markdown, display

# Set max width for results dataframe
pd.set_option('display.max_colwidth', 200)

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Define model

In [3]:
llm = load_llm()

# RAG Semantic

## Load vector store

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

corpus_path = '../data/processed/preprocessed_corpus.csv'
vector_path = "../data/processed/vector_store"
docs = csv_loader(corpus_path)

if not os.path.exists(vector_path):
    build_vectorstore(docs, vector_path, embeddings)
    print(f"Saved vector store to {vector_path}")
    
vectorstore = FAISS.load_local(
    vector_path, embeddings, allow_dangerous_deserialization=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Retrieval

In [5]:
vector_retriever = semantic_retriever(vectorstore)
query = "what is the best soap"
format_docs_to_df(vector_retriever.invoke(query))

,Product ASIN,Title,Rating,Review
0,B0716PQVP2,Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool,5.0,Must have with bars of soap ! You will love !
1,B08DV37PZV,"Palmolive Ultra Original Dish Liquid, 102 fl. oz. - 2 Pack",5.0,"That ""blue"" dish soap is more difficult to rinse off. I like Palmolive because it cleans well and rinses off easily."
2,B001HDZT7I,"Travelon Hand Soap Toiletry Sheets, 50-Count",3.0,Although this had decent reviews when I researched I learned very quickly that once these are wet they are not usable. The sheet size maybe can cover hand washing but they clump up and there isn't...
3,B00O92P0KU,Lemon Essential Oil 4 Oz - 5x Extra Strength 100% Pure & Natural Therapeutic Grade - Cold Pressed PREMIUM QUALITY Oil from Italy,5.0,"The Radha lemon essential oil is wonderful – a big bottle of very strong smelling lemon oil (not a sweet lemon, but a very ‘true’ lemon scent). I use a LOT of lemon oil for making homemade dish so..."
4,B08W2GY3KV,Beautywin Soft Silicone Bath Brush，Baby Shower Exfoliating Body Scrubber Cleaning Massage Tool Bathroom Products for Adult Pet Clean Tools (Yellow),1.0,Soap just falls right out it feel nice but ur soap doesn’t stay in it at all I filled it up put it down the grave it again all of soap was where it was sitting


## Pipeline

In [6]:
def text(rag_chain, query):
    display(Markdown(str(rag_chain.invoke(query))))

In [7]:
rag_chain = initialize_rag_chain(vector_retriever, llm, prompt)
text(rag_chain, query)

Based on the reviews provided, it seems that there isn't a straightforward answer to what the "best" soap is. However, I can provide some insights based on the customer reviews:

For bar soap, the reviewer of the Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch (ASIN: B0716PQVP2) mentions that it is a "must-have" with bars of soap. 

For liquid soap, the reviewer of the Palmolive Ultra Original Dish Liquid (ASIN: B08DV37PZV) suggests that it is a good option because it cleans well and rinses off easily.

It's worth noting that the reviewer of the Travelon Hand Soap Toiletry Sheets (ASIN: B001HDZT7I) wasn't very satisfied with this product and preferred using bar or liquid soap.

As for making homemade soap, the reviewer of the Lemon Essential Oil (ASIN: B00O92P0KU) has a recipe for homemade dish soap that uses lemon essential oil, borax, grated soap, and water. They mention that this homemade soap is effective and leaves a clean scent.

In terms of the best soap, it ultimately depends on personal preferences and what you're looking for in a soap. If you prefer bar soap, the reviewer of the Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch suggests that it is a good option. If you prefer liquid soap, the reviewer of the Palmolive Ultra Original Dish Liquid suggests that it is a good option.

In [8]:
rag_chain = initialize_rag_chain(vector_retriever, llm, prompt_no_instruct)
text(rag_chain, query)

Based on the customer reviews provided, it seems like the best soap for a specific use case can be determined by reading the reviews carefully. 

For a dish soap, Palmolive Ultra Original Dish Liquid (Product ASIN: B08DV37PZV) seems to be a popular choice among customers, as it cleans well and rinses off easily. 

For making homemade dish soap, a customer review mentions using a recipe that includes lemon essential oil. They have a positive review for the Radha lemon essential oil (Product ASIN: B00O92P0KU), and they mention using it to make a homemade dish soap which is effective and gentle on skin.

For body and hand soap, the Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch (Product ASIN: B0716PQVP2) seems to be a good option as it is a soap saver pouch that helps keep soap clean and dry when not in use.

As for the best soap overall, the Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch (Product ASIN: B0716PQVP2) seems to be a good option due to its high rating of 5.0/5.0 and the positive review from a customer who mentions that it's a "must-have" with bars of soap.

In [9]:
rag_chain = initialize_rag_chain(vector_retriever, llm, prompt_concise)
text(rag_chain, query)

Based on the reviews, the best soap is likely made at home using lemon essential oil (Product ASIN: B00O92P0KU) and a recipe similar to the one described in the review. This homemade soap has been praised for its effectiveness in cutting grease and leaving a clean scent.

# Hybrid RAG: Semantic Search + BM25

### BM25 Retriever

In [10]:
keyword_retriever = bm25_retriever(docs)

# preprocess query so it's consistent with corpus preprocessing
query = preprocess_query(query)

# Retriever invoke returns the top_k docs
format_docs_to_df(keyword_retriever.invoke(query))

,Product ASIN,Title,Rating,Review
0,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",4.0,A single sheet is good for a pair of underwear. The soap dissolves quickly but the water does not feel too soapy so not sure about the quality of cleaning. Best is to let the clothes sit in the so...
1,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",3.0,"If you MUST save on space, consider these laundry soap sheets. I will be honest, though... hotel bar soap and baby wash made my clothes smell (and therefor feel) cleaner to me when doing laundry ..."
2,B00OPC61V6,"Dial Mountain Fresh Antibacterial Deodorant Soap, 4 oz, 6 Count",5.0,"I use Dial soap because it does the best job of fighting odor, particularly during the summer when it's so hot and humid in the South. Leaves me feeling clean and fresh, rinses well, nice fragran..."
3,B001CGOPZM,"Travelon Hand Soap Toiletry Sheets, 50-Count",5.0,I bought this to keep with me on my recent trip to several Asian countries. Those countries are notorious for not providing soap in restrooms. This travel soap worked perfectly for my needs. I am ...
4,B0773H89Y6,"- Soap Dispensing Dish Brush -,Multipurpose Household Cleaning Brushes,Sink Brush by Samshow",1.0,The previous reviews were either really positive or not. Gave this a try. Loaded up the handle with soap then laid it down in the sink. Came back in a few minutes and there was soap at the bott...


## Ensemble BM25 + Semantic Retriever

In [11]:
# Invoke to get combined results
ensemble_retriever = hybrid_retriever(keyword_retriever, vector_retriever)
format_docs_to_df(ensemble_retriever.invoke(query))

,Product ASIN,Title,Rating,Review
0,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",4.0,A single sheet is good for a pair of underwear. The soap dissolves quickly but the water does not feel too soapy so not sure about the quality of cleaning. Best is to let the clothes sit in the so...
1,B001CGOPZM,"Travelon Hand Soap Toiletry Sheets, 50-Count",5.0,nice product
2,B004UJHY74,Sekkisei cream wash,5.0,Smells nice and soapy. Creamy. Does a great job. This will be a regular purchase.
3,B0716PQVP2,Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool,5.0,Must have with bars of soap ! You will love !
4,B08W2GY3KV,Beautywin Soft Silicone Bath Brush，Baby Shower Exfoliating Body Scrubber Cleaning Massage Tool Bathroom Products for Adult Pet Clean Tools (Yellow),1.0,Soap just falls right out it feel nice but ur soap doesn’t stay in it at all I filled it up put it down the grave it again all of soap was where it was sitting
5,B000Y0CL8K,"Travelon Hand Soap Toiletry Sheets, 50-Count",3.0,"If you MUST save on space, consider these laundry soap sheets. I will be honest, though... hotel bar soap and baby wash made my clothes smell (and therefor feel) cleaner to me when doing laundry ..."
6,B00OPC61V6,"Dial Mountain Fresh Antibacterial Deodorant Soap, 4 oz, 6 Count",5.0,"I use Dial soap because it does the best job of fighting odor, particularly during the summer when it's so hot and humid in the South. Leaves me feeling clean and fresh, rinses well, nice fragran..."
7,B001CGOPZM,"Travelon Hand Soap Toiletry Sheets, 50-Count",5.0,I bought this to keep with me on my recent trip to several Asian countries. Those countries are notorious for not providing soap in restrooms. This travel soap worked perfectly for my needs. I am ...
8,B0773H89Y6,"- Soap Dispensing Dish Brush -,Multipurpose Household Cleaning Brushes,Sink Brush by Samshow",1.0,The previous reviews were either really positive or not. Gave this a try. Loaded up the handle with soap then laid it down in the sink. Came back in a few minutes and there was soap at the bott...


In [12]:
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt)
text(rag_chain, query)

It seems like you're looking for a good soap. 

Based on the reviews, I'd recommend the "Dial Mountain Fresh Antibacterial Deodorant Soap" (ASIN: B00OPC61V6) and the "Sekkisei cream wash" (ASIN: B004UJHY74). 

The Dial soap is rated 5.0/5.0 and leaves the user feeling clean and fresh, while the Sekkisei cream wash is also rated 5.0/5.0 and is described as smelling nice and soapy, doing a great job at cleaning.

In [13]:
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt_no_instruct)
text(rag_chain, query)

Based on the reviews provided, it appears that the following soaps are highly rated by customers:

1. **Sekkisei Cream Wash (ASIN: B004UJHY74)** - This soap has received a 5.0/5.0 rating from one customer, praising its nice smell, soapy and creamy texture, and ability to do a great job cleaning.

2. **Dial Mountain Fresh Antibacterial Deodorant Soap (ASIN: B00OPC61V6)** - This soap has received a 5.0/5.0 rating from one customer, praising its ability to fight odor, leave them feeling clean and fresh, and rinse well.

3. **Travelon Hand Soap Toiletry Sheets (ASIN: B001CGOPZM)** - This soap has received multiple high ratings, including a 5.0/5.0 rating from one customer, praising its effectiveness, ability to keep hands clean, and safety features for handling contact lenses.

These soaps have received high ratings for their effectiveness, nice scents, and ability to clean well. However, it's essential to note that individual preferences may vary, and what works well for one person may not work as well for another.

If you're looking for a good soap, I would recommend considering these options. However, if you have specific preferences or needs, please let me know, and I'll be happy to help you find the best soap for you.

In [14]:
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt_concise)
text(rag_chain, query)

Based on the reviews, the following soaps are highly rated:

1. Dial Mountain Fresh Antibacterial Deodorant Soap (ASIN: B00OPC61V6) - 5.0/5.0
2. Sekkisei cream wash (ASIN: B004UJHY74) - 5.0/5.0
3. Travelon Hand Soap Toiletry Sheets (ASIN: B001CGOPZM) - 4.0/5.0 and 5.0/5.0 (two separate reviews)

# Qualitative Evaluation for Hybrid RAG Workflow

In [15]:
# query list from Milestone 1
queries = ["Wet wipes",
           "Bar Soap",
           "Small hair dryer",
           "The best air humidifer with essential oil",
           "mineral sunscreen for babies",
           "hair spray that last more than 6 hours",
           "Best Vitamins or supplements to take for pregnant women",
           "equipments for stretching at home",
           "sunrise lamp that will help me to wake up in the morning",
           "Something to relieve my back pain"]

In [16]:
# use original prompt
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt)
for q in queries:
    display(Markdown(f"##QUERY: {q}\n"))
    text(rag_chain, q)
    display(Markdown("### Sources"))
    retrieved_docs = ensemble_retriever.invoke(q)
    df = format_docs_to_df(retrieved_docs)
    display(df)

##QUERY: Wet wipes


Based on the reviews, it seems that some wet wipes are thick and effective for cleaning. 

- The Rolhei 75% Ethanol Wet Wipe (B08HVXPVWV) is described as "thick" and presumably effective.
- The Rinse Free Sponge Bath Wipes (B095WR73ZP) are also described as "thick" and able to clean at least the feet, but the reviewer found the package size to be inconvenient for the task.
- Pre-Moistened Lens Cleaning Wipes (B07414X6SL) are not specifically mentioned as being thick, but rather as a pain to use because they require one wipe for cleaning and one for drying.

It's also worth noting that some wet wipes are more effective than others for specific tasks, such as eyelid cleansing. The Systane Lid Wipes (B00HHXFQSY) have a high rating from multiple reviewers who found them to be effective for their purposes.

Overall, the effectiveness of wet wipes seems to depend on the specific task and the product in question.

### Sources

,Product ASIN,Title,Rating,Review
0,B08HVXPVWV,Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (200 Wipes),5.0,that the wipes are thick and not thin.
1,B095WR73ZP,Rinse Free Sponge Bath Wipes (30-pack) | Extra Large & Thick No Rinse Waterless Body Wipe Sponges - Disposable Shower Cleaning Wash Cloths for Kids Adults Surgery Elder Care Gym Travel & More (1 P...,3.0,"I didn’t expect much from these, and I didn’t get much. Thankfully, I didn’t get them as a bath/shower substitute so I have to admit, that I was able to get SOME use from them.<br /><br />There wa..."
2,B07414X6SL,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry Wipes 300 pcs, for Lens Eyeglasses Glasses Screen iPhone Cell Phone, Remove smudges and Dirt Effectively, no More Scratches Streaks Residue (300)",1.0,One towel to clean and one to dry. What a pain.
3,B0878RRKZ2,"Eyliden Flat Mop and Bucket Set, Wet & Dry Mops for Floor Cleaning, with Stainless Steel Handle + 2-Reusable Microfiber Pads, Home Kitchen Office Cleaner for Hardwood, Tiles, Laminate (White+Red)",5.0,"Cleaning isn't fun by any means, but having tools to make it less of a chore is great. This is a nice set to help keep your floors clean. There are two sides, one for your wash water and one that ..."
4,B00DDMJ6NE,"Pampers Baby Fresh Water Baby Wipes 3X Pop-Top Packs, 192 Count",4.0,Decent quality but I'll stick with huggies wipes.
5,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",4.0,I’m not impressed.
6,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",5.0,Very good product
7,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",4.0,The wipes are large and easy to use.
8,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",4.0,My blepharitis was making me crazy. These are easy to use and work like they say.
9,B00HHXFQSY,"Systane Lid Wipes - Eyelid Cleansing Wipes - Sterile, Count of 32",5.0,My husband has chronic dry eyes. He loves these wipes; says nothing he has used helps as much.


##QUERY: Bar Soap


...

,Product ASIN,Title,Rating,Review
0,B00BYG3OWK,Naturebright L6060 Per2 Led Daylight Lamp,4.0,"I tried the Phillips previously (see review there) but just couldn't get it to work. The NatureBright is easy to figure out, the controls make a lot more sense and you can easily see that it's se..."
1,B00BYG3OWK,Naturebright L6060 Per2 Led Daylight Lamp,3.0,"This functions well as daylight alarm. It's best feature is that the LED's arc out over the bed and shine down, as opposed to other models which shine straight out horizontally, which means you ha..."
2,B000N5K1HO,"C&A Scientific Alcohol Lamp, Metal, Wickless, 100ml (97-5320)",4.0,The upside is that this is a very well made little lamp.<br />it's easy to start and produces a hot and steady flame.<br />If you need a light duty flame hot enough to heat a test tube or steriliz...
3,B00BYG3OWK,Naturebright L6060 Per2 Led Daylight Lamp,4.0,A+ However a bit difficult to program.
4,B01NH32YAL,NYKKOLA Sleep Restoration Natural Silk Sleep Mask & Blindfold - Single Strap Super-Soft Silk Eye Mask,3.0,"Is super comfortable and is great at keeping the light out.... at first. after using and wearing this for a few weeks, the fabric where my nose goes started to stay permanently curled up, letting ..."
5,B003DS209K,"5-hour ENERGY Shot, Regular Strength Orange, 1.93 oz., 24 pack",5.0,I got what I needed to wake up and get out to my commute while it's still dark. I am the poster child for &#34;not a morning person.&#34; This makes it happen.
6,B07H4J2CVH,"Sleep Mask for Women and Men,3D Contoured Eye Mask for Sleeping Mask Eye Cover,Lightweight Sleep Masks Blindfolds Eye Shade for Kids Girls Travel,2 Pack Black",5.0,"Very comfortable, which surprised me! I love that it doesn't press against your eyes. I also suffer from extremely dry eyes, especially in the morning when I wake up, due to multiple fans running ..."
7,B003TPXYF6,Malem Wireless Bedwetting Alarm System,5.0,My son is almost 9yrs old. He had been wet EVERY single night since birth. He would often wet through pull-ups. It was really starting to effect his self-esteem. Sleep-overs and scout trips wer...
8,B000F95A4W,"Verilux Original Natural Spectrum Deluxe Floor Lamp, Ivory",5.0,I had one of these lamps for several years and liked for reading. As I consider it a fairly expensive lamp I was disappointed when the switch quit working. I didn't get a response when I contact...


##QUERY: Something to relieve my back pain


Based on the reviews, I would recommend the following products to help relieve back pain:

1. Genericb Back Massage Stretcher Arch Magic Message Stretcher Back Stretcher Lumbar Support Device (B08NVFSSH8) - a back stretcher that provides lumbar support and relief from lower and upper back pain.
2. Back Massager, Aptoco Back Stretcher for Sciatica Relief (B074QLYVF9) - a back stretcher that helps relieve sciatica pain and provides back massage.
3. ZSZBACE Posture Corrector Back Brace (B0787TBYP9) - a posture corrector back brace that helps relieve back pain by aligning the spine.
4. CareU Coccyx Orthopedic Portable Comfort Memory Foam Seat Cushion (B019O0QNT8) - a seat cushion that helps relieve hip and sciatica pain by providing support and relief to the coccyx area.

These products have received positive reviews from customers who have experienced relief from back pain. However, please note that individual results may vary, and it's always best to consult with a healthcare professional before trying any new product.

### Sources

,Product ASIN,Title,Rating,Review
0,B08NVFSSH8,"Genericb Back Massage Stretcher Arch Magic Message Stretcher Back Stretcher Lumbar Support Device, Lower and Upper Back Pain Relief Relax Mate Spine Pain Relief Chiropractic",4.0,A little to stiff for sore back muscles. need to use a soft pad with it
1,B074QLYVF9,"Back Massager, Aptoco Back Stretcher for Sciatica Relief, Herniated Disc/Spinal Stenosis-Back Massage Stretcher, Back Stretching Device",4.0,A little uncomfortable for me (not very flexable) but my wife loves it and uses it multiple times a day.
2,B00VRAPGM4,"Acupuncture Mat and Pillow Set-Relieve Your Stress, Back, Neck, and Sciatic Pain(99% Cotton Fabric, Plastic Spikes and Foam core) Green",5.0,Really helped with tension in my shoulders!
3,B019O0QNT8,"CareU(™) Coccyx Orthopedic Portable Comfort Memory Foam Seat Cushion with Non Slip Cover- Helps Relieve Kyphotic, Hip and Sciatica Pain (Standard Size 17.7x15 Inch)",5.0,I'm sitting on it right now. love it
4,B075GWYBD9,"NatraCure Microwaveable Moist Heat Wrap (w/Clay Beads) - (Hot Therapy Pack and Heating Pain Relief Pad for Neck, Shoulders, Body)",5.0,I love this product! I have alot of muscle pain in my shoulders and this has allowed me to relax. Would definitely recommend!
5,B08Z2SDRL2,"Neck Stretcher Cervical Neck Traction Device Over Door for Home Use,Portable Neck Traction for Neck Pain Relief, Physical Therapy AIDS for Neck Spine Decompressor (Black)",5.0,First time user. It took me a while a find out how to assemble them. After some adjustments I find a way to use it. And it does helps relieve my neck pain.
6,B01MR6QQBB,Shoulder Wrap Gel Ice Hot Cold Pack for Shoulder Injury Pain Relief Therapy Rotator Cuff Rheumatoid Arthritis Treatment Osteoarthritis Bursitis Tendinitis AC Joint Sports Injuries,5.0,"I use it alot, great product, helps relieve my muscle pain."
7,B0787TBYP9,"ZSZBACE Posture Corrector Back Brace for Men and Women- Relieve Back Pain, Align Spain, Correct Kyphosis (XXL)",5.0,It help with my back pain
8,B015MUK59S,Korean Red Ginseng Patch Powerstrip Energy Pain Relief - 20 Patches,5.0,These patches really do relieve the pain. They can be cut to whatever size you need.They stick extremely well.
9,B07Y8VTPJ8,"Real Time Pain Relief George Foreman's Knockout Formula, 1 Oz GoPak",5.0,This stuff does exactly what it says it will do. Relieve pain and make you feel better. Happy!
